In [1]:
import nltk
import re
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.chunk import ne_chunk
from nltk.corpus import words as nltk_words

In [2]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## E-Commerce NER: Sample Product Reviews

In [3]:
reviews = [
    "I bought the iPhone 15 from Apple and the camera quality is amazing. "
    "It cost me $999 at the New York store.",

    "Samsung Galaxy S24 has excellent battery life but the display is dim. "
    "Bought it from Amazon in California for $899.",

    "My Dell XPS 15 arrived from Dell warehouse in Texas. "
    "The build quality is great and price was ₹145,000.",

    "Nike Air Max shoes have great comfort and design. "
    "Purchased from Nike outlet in Chicago for €120."
]

## Define Extraction Rules

In [4]:
BRANDS = [
    "apple", "samsung", "dell", "nike", "sony", "lg", "hp",
    "lenovo", "bose", "adidas", "microsoft", "google", "amazon"
]

FEATURES = [
    "battery life", "camera quality", "display", "build quality", "comfort",
    "design", "performance", "sound quality", "screen size", "storage",
    "price", "weight", "durability"
]

PRICE_PATTERN = re.compile(
    r"\$[\d,]+(?:\.\d{2})?|₹[\d,]+(?:\.\d{2})?|€[\d,]+(?:\.\d{2})?"
)

PRODUCT_PATTERNS = [
    re.compile(r"\b(?:iphone\s*\d+|galaxy\s*\w+|dell\s*xps\s*\d+|macbook\s*\w+|air\s*max)\b", re.IGNORECASE),
    re.compile(r"\b(?:pixel\s*\d+|surface\s*\w+|thinkpad\s*\w+|galaxy\s*\w+|iphone)\b", re.IGNORECASE),
]

## NLTK NER: Extract Organizations and Locations

In [5]:
def get_entities_ne_chunk(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    chunks = ne_chunk(pos_tags)

    orgs = []
    locs = []

    for chunk in chunks:
        if hasattr(chunk, 'label'):
            entity_type = chunk.label()
            entity_text = " ".join(c[0] for c in chunk.leaves())
            if entity_type == 'ORGANIZATION':
                orgs.append(entity_text)
            elif entity_type in ['GPE', 'LOCATION', 'FACILITY']:
                locs.append(entity_text)

    return orgs, locs

## Custom E-Commerce Entity Extraction

In [6]:
def extract_ecommerce_entities(text, orgs, locs):
    lower_text = text.lower()

    brands = list(set([
        b for b in BRANDS
        if re.search(rf"\b{re.escape(b)}\b", lower_text)
    ]))

    products = []
    for pat in PRODUCT_PATTERNS:
        matches = pat.findall(text)
        products.extend([m.strip().title() for m in matches])

    features = [f for f in FEATURES if f.lower() in lower_text]

    prices = PRICE_PATTERN.findall(text)

    return brands, products, features, prices

## Combine Results into Structured Frame

In [7]:
def analyze_review(text):
    orgs, locs = get_entities_ne_chunk(text)
    brands, products, features, prices = extract_ecommerce_entities(text, orgs, locs)

    frame = {
        "ORGANIZATION": orgs or ["UNKNOWN"],
        "LOCATION": locs or ["UNKNOWN"],
        "BRAND": brands or ["UNKNOWN"],
        "PRODUCT": products or ["UNKNOWN"],
        "FEATURE": features or ["UNKNOWN"],
        "PRICE": prices or ["UNKNOWN"]
    }
    return frame

## Run Analysis on All Reviews

In [8]:
for i, review in enumerate(reviews, 1):
    frame = analyze_review(review)
    print(f"\nReview {i}: {review.strip()}")
    print("-" * 60)
    for key, values in frame.items():
        print(f"{key:<15}: {', '.join(values)}")


Review 1: I bought the iPhone 15 from Apple and the camera quality is amazing. It cost me $999 at the New York store.
------------------------------------------------------------
ORGANIZATION   : iPhone
LOCATION       : Apple, New York
BRAND          : apple
PRODUCT        : Iphone 15, Iphone
FEATURE        : camera quality
PRICE          : $999

Review 2: Samsung Galaxy S24 has excellent battery life but the display is dim. Bought it from Amazon in California for $899.
------------------------------------------------------------
ORGANIZATION   : UNKNOWN
LOCATION       : Amazon, California
BRAND          : amazon, samsung
PRODUCT        : Galaxy S24, Galaxy S24
FEATURE        : battery life, display
PRICE          : $899

Review 3: My Dell XPS 15 arrived from Dell warehouse in Texas. The build quality is great and price was ₹145,000.
------------------------------------------------------------
ORGANIZATION   : Dell, Dell
LOCATION       : Texas
BRAND          : dell
PRODUCT        : De